# z709 — LightGBM par ⟨cliente, producto⟩, horizonte 2 genérico

Sucesor del z702/z705/. Esquema de la pizarra: densa → escalado + FE → clase `t+2` →
clusters DTW → Optuna y un modelo por cluster → walk-forward → predict desescalado → submit.

## La ley inviolable

**Lo único que mira el futuro es la clase** (`tn(t+2)`). Features y escalado salen de `<= t`, el
mes donde estás parado. No se declara: **se verifica**. La celda 4 recalcula todo el FE sobre la
historia truncada en T y exige que las filas `periodo <= T` no cambien ni un decimal; si algo
cambia, **aborta**. Ya cazó tres fugas reales: el `rolling` del z702 que se derramaba de la serie
anterior, un `cum_sum` por cliente que seguía el orden de filas, y el índice estacional cuando lo
escribí sin el `join_asof`.

Y una fuga que ese test **no** puede ver, porque no está en el FE sino en el corte: el fold debe
entrenar con `periodo <= corte − h`. Con `periodo <= corte`, las filas de los últimos h meses
traen como clase justamente el mes que se valida. Hay un assert aparte.

## Lo que cambia acá

**1. El modelo tiene que servir para cualquier mes, no para febrero.** La competencia pide
febrero, pero un modelo que sólo funciona en febrero está sobreajustado al fold. Se valida en
**siete anclas repartidas por todo el año**, y el WAPE se reporta mes por mes.

Esto no es prolijidad: el sell-in tiene una estacionalidad fuerte y asimétrica. Índice medido
sobre los 780 productos (media del mes / media global):

```
  marzo 1.162   OCTUBRE 1.153   septiembre 1.076   mayo 1.080   junio 1.027
  noviembre 1.047   abril 0.991   julio 0.955   agosto 0.942
  febrero 0.884   DICIEMBRE 0.844   enero 0.839
```

**Octubre es el segundo pico del año y diciembre es el piso.** Con horizonte 2, el modelo parado
en octubre —viendo un nivel altísimo, porque en octubre se carga para cerrar el año— tiene que
predecir diciembre, que es un 27 % más chico. Sin corrección estacional sobre-predice ~37 %. Una
validación sólo contra febrero no ve nunca ese error. Por eso el ancla **201812** (origen
octubre) está entre los folds.

**2. FE estacional del mes objetivo.** Es lo que le faltaba al z708. Tres features nuevas, todas
causales:

- `mes_objetivo` — el mes que se predice, `(mes + 2)`, como categórica.
- `estacional_prod_objetivo` — cuánto pesa históricamente ese mes para *ese producto*, relativo a
  su propio promedio, usando sólo ocurrencias `<= t` (`join_asof` hacia atrás). Es la corrección
  que evita el error octubre→diciembre.
- `avance_anual_prod` — acumulado del año hasta t contra el mismo tramo del año anterior. Proxy
  del empuje por objetivos: quien viene atrasado respecto de su año pasado, aprieta.

**3. Sin canaritos.** El set de features se fija por criterio y por lo que se repitió en las dos
corridas del selector, no por una barrera de ruido. El canarito pregunta "¿tiene *algo* de
señal?", que con millones de filas casi todo contesta que sí, y encima mide ganancia *in-sample*.
Se sacaron las que quedaron al fondo en ambas corridas.

**4. El peso `s^(p−1)`.** Con `init_score = log(s)` y Tweedie (link log) el óptimo de una hoja es
`Σ s^(1−p)y / Σ s^(2−p)`: el ratio `y/s` ponderado por `s^(2−p)`, no por `s`. Con `p=1.3` una fila
de 300 t pesa 55 y una de 1 kg pesa 0,002 — las millones de filas chicas (80 % con clase cero)
arrastran la hoja hacia abajo, y ése es el 6,5 % de tonelaje que le faltaba al z705. Con
`weight = s^(p−1)` el peso efectivo queda en `s` y la hoja vale **exactamente `Σy/Σs`**.
Verificado numéricamente: sesgo 0,00 % para todo `p`.

**5. Clusters DTW a nivel producto.** Cada producto entra entero en un cluster, así que
`Σ_p |e_p|` se descompone exacto y tunear cada cluster por separado optimiza la métrica global.
En el z702 los clusters eran por par y el WAPE se medía sobre sumas parciales del producto.

## Reanudable

Checkpoints atómicos en el bucket (densa, FE, clusters por corte, estudio de Optuna,
hiperparámetros). Si la VM se cae, volvés a correr todo y retoma.

Estimación sobre 17,17 M filas: densa ~10 min, FE ~15, DTW por corte ~25, Optuna 4–5 h,
reporte de 7 anclas ~50 min, A/B estacional ~50 min, final ~45 min. **Total 7–9 h.**

In [ ]:
%pip install -q polars duckdb lightgbm optuna dtaidistance scikit-learn scipy pandas numpy pyarrow

In [ ]:
# ruff: noqa: E402
import gc
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAY_OPTUNA = True
except Exception:
    HAY_OPTUNA = False

try:
    from dtaidistance import dtw
    DTW_C = dtw.try_import_c()
except Exception:
    dtw, DTW_C = None, False

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/.drive")
    os.makedirs("/content/buckets", exist_ok=True)
    if not os.path.islink("/content/buckets/b1"):
        os.symlink("/content/.drive/My Drive/labo3", "/content/buckets/b1")
    os.environ["LABO3_BUCKET"] = "/content/buckets/b1"

N_CORES = os.cpu_count() or 8


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env:
        p = Path(env); p.mkdir(parents=True, exist_ok=True); return p
    for c in ("/home/jupyter/buckets/b1", "/content/buckets/b1", "~/buckets/b1"):
        p = Path(c).expanduser()
        if p.exists():
            return p
    p = Path.cwd() / "bucket"; p.mkdir(parents=True, exist_ok=True); return p


BUCKET = resolver_bucket()


def escribir_atomico(fn, destino: Path):
    """Escribe por un temporal y copia. El rename sobre un bucket montado por FUSE no es confiable."""
    tmp = Path("/tmp") / (destino.name + ".tmp")
    fn(tmp)
    import shutil
    shutil.copyfile(tmp, destino)
    tmp.unlink(missing_ok=True)


def meses_entre(a: int, b: int) -> int:
    return (b // 100 - a // 100) * 12 + (b % 100 - a % 100)


def periodo_menos(p: int, k: int) -> int:
    a, m = divmod(p, 100)
    t = a * 12 + (m - 1) - k
    return (t // 12) * 100 + (t % 12) + 1


print(f"bucket: {BUCKET} | cores: {N_CORES} | optuna: {HAY_OPTUNA} | DTW en C: {DTW_C}")

## 1 — Palancas

In [ ]:
PARAM = {
    "experimento": "z709_lgbm_par_estacional",
    "kaggle_competition": "labo-iii-2026-ba",
    "modo_test": False,          # True = smoke test con pocos productos

    "horizonte": 2,
    "periodo_inferencia": 201912,
    "periodo_objetivo": 202002,

    # --- escalado (relativo al mes donde estás parado, causal) ---
    "escala": "media_expandida",   # media_expandida | media_movil_12 | mediana_movil_12
    "piso_escala": 1e-3,

    # --- objetivo ---
    # 'tweedie'  : init_score=log(s) + peso s^(p-1)  -> la hoja vale Sum(y)/Sum(s)   [recomendado]
    # 'agrupado' : objetivo custom cuyo gradiente es el signo del error AGREGADO por producto,
    #              que es literalmente el numerador de WAPE. Experimental: A/B en la celda 12.
    "objetivo": "tweedie",
    "tweedie_power": 1.3,          # tuneado por Optuna dentro de [1.1, 1.6]
    "usar_peso_escala": True,      # el s^(p-1). Ponelo en False para ver el sesgo del z705.

    # --- features (24). Criterio, no canarito: se sacaron las que quedaron al fondo en las
    #     dos corridas del selector (meses_desde_compra, racha_max_12, tasa_actividad_12,
    #     croston_intervalo, sku_size) y se sumaron las tres estacionales nuevas. ---
    "features": [
        # nivel y ritmo del par, todo relativo a s(t)
        "nivel_rel", "lag1_rel", "lag2_rel", "lag3_rel", "lag12_rel",
        "rmean3_rel", "rmean12_rel", "rstd3_rel", "tendencia_3_12",
        # actividad y escala
        "ventas_ult12", "log_escala",
        # calendario y estacionalidad del MES OBJETIVO  <- lo nuevo
        "mes", "mes_objetivo", "estacional_prod_objetivo", "avance_anual_prod",
        # contexto de producto y cliente
        "tn_prod_rel", "share_par_en_prod", "share_par_en_cli", "cliente_rank_rel",
        "tendencia_prod_3_12", "share_prod_en_cat3", "stock_idx_familia_3",
        # estaticas
        "cat3", "brand",
    ],
    "categoricas": ["cat3", "brand", "mes", "mes_objetivo"],

    # --- clusters DTW (nivel producto: el WAPE se descompone exacto) ---
    "usar_clusters": True,
    "cl_lista_k": [4, 5, 6, 7, 8],
    "cl_ventana": 24,
    "cl_banda": 3,
    "cl_balance_min": 0.03,        # ningun cluster con menos del 3% de los productos

    # --- validacion walk-forward repartida en el ano ---
    # Optuna usa 3 anclas que cubren los tres regimenes: diciembre (origen OCTUBRE, el caso
    # dificil), febrero (el target) y agosto (mes flojo). El reporte final mide en las 7.
    "anclas_optuna": [201812, 201902, 201908],
    "anclas_reporte": [201806, 201809, 201812, 201902, 201905, 201908, 201910],
    "peso_febrero": 1.5,   # el target es febrero, pero el modelo tiene que servir todo el ano

    # --- entrenamiento ---
    "max_bin": 1023,
    "techo_arboles": 900,
    "n_trials": 15,
    "submuestreo_dormidos": 0.25,
    "semillas_ensemble": [102191, 314159, 777773],

    # --- calibracion y control ---
    "ab_estacional": True,   # +1 pasada por ancla: mide si el FE estacional realmente paga
    "lambda_shrink": 0.7,
    "banda_nivel": [28400, 30600],  # feb-2020 plausible; fuera de esto se avisa fuerte
    "submit": True,
    "sufijo": "",

    # Versiona los checkpoints. Si cambia la densa, el FE o la lista de features, SUBILA: los
    # checkpoints viejos quedan ignorados en vez de reusarse en silencio con otra logica.
    # v1 = densa solo con pares observados (10,5M). v2 = cartesiano de la catedra (17,2M).
    "version_datos": "v2_cartesiano",
}

if PARAM["objetivo"] == "agrupado":
    # sin filas no se puede agregar bien, y el peso s^(p-1) alinea una perdida por fila que
    # este objetivo ya no usa
    PARAM.update(submuestreo_dormidos=1.0, usar_peso_escala=False)
    print("objetivo AGRUPADO: se apagan submuestreo de dormidos y peso s^(p-1)")

if PARAM["modo_test"]:
    PARAM.update(n_trials=3, techo_arboles=120, semillas_ensemble=[102191],
                 anclas_optuna=[201902], anclas_reporte=[201812, 201902],
                 cl_lista_k=[3, 4], submit=False)

DIR_RAW = BUCKET / "datasets"
DIR_EXP = BUCKET / "exp" / (PARAM["experimento"] + PARAM["sufijo"])
DIR_CK = DIR_EXP / "ck" / PARAM["version_datos"]
DIR_OUT = DIR_EXP / "submits"
for d in (DIR_RAW, DIR_EXP, DIR_CK, DIR_OUT):
    d.mkdir(parents=True, exist_ok=True)

FEATS = PARAM["features"]
CATEGORICAS = [c for c in PARAM["categoricas"] if c in FEATS]
print(f"experimento: {DIR_EXP}")
print(f"checkpoints: {DIR_CK}  (version {PARAM['version_datos']})")
print(f"features ({len(FEATS)}): {FEATS}")

## 2 — Densa zero-fill (criterio cátedra, **producto cartesiano**)

`todos los clientes × todos los productos × todos los periodos`, recortado a la vida del producto
(`min..max` observado) y a partir del primer periodo del cliente. **17.173.448 filas.**

Es el `tb_zeroes` del `z601` de la cátedra, escrito como `CROSS JOIN` + `LEFT JOIN` en vez de
`NOT EXISTS` + `UNION` porque es mucho más rápido y da lo mismo.

**Por qué el cartesiano y no sólo los pares que comerciaron** (que serían 10.469.949, un 61 %):
no es sólo fidelidad al criterio. Quedarse con los pares observados obliga a definir el universo
de filas con `DISTINCT (cliente, producto)` sobre **toda** la historia, futuro incluido: en un
fold cortado en 201810 entrarían pares cuyo primer intercambio ocurre recién en 201905. Las
features no se contaminan —esas filas son ceros— pero la *composición del train* pasa a depender
del futuro, y eso un deploy real no lo puede saber. El cartesiano es libre de fuga por
construcción. Además permite predecir la activación de pares nuevos, que es el **1,6 % del
tonelaje mensual** (1,9 % en los últimos seis meses).

Los 6,7 M de filas de más son casi todas de pares dormidos, así que el submuestreo al 25 % y el
peso `s^(p−1)` las dejan pesando poco: el costo real es de orden +20 %, no +64 %.

Los meses quedan **contiguos**, y hay un assert que lo verifica en vez de darlo por sentado: sin
contigüidad, `shift(k)` deja de ser el lag de k meses y todos los lags mienten.

In [ ]:
def descargar(a: str):
    dst = DIR_RAW / a
    if dst.exists():
        return
    url = f"https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{a}"
    subprocess.run(["wget", "-q", url, "-O", str(dst)], check=True)


for _a in ("sell-in.txt.gz", "tb_productos.txt", "product_id_apredecir201912.txt"):
    descargar(_a)

PROD_TARGET = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt",
                          separator="\t")["product_id"].cast(pl.Int64).to_list()

CK_DENSA = DIR_CK / "densa.parquet"
if CK_DENSA.exists():
    densa = pl.read_parquet(CK_DENSA)
    print(f"densa desde checkpoint {CK_DENSA.parent.name}: {densa.height:,} filas")
else:
    t0 = time.time()
    con = duckdb.connect()
    con.execute("PRAGMA threads=%d" % N_CORES)
    con.execute(f"""CREATE OR REPLACE VIEW crudo AS
        SELECT customer_id, product_id, periodo, sum(tn) AS tn
        FROM read_csv('{DIR_RAW / "sell-in.txt.gz"}', delim='\t', header=true)
        GROUP BY 1,2,3""")
    filtro = ""
    if PARAM["modo_test"]:
        filtro = f"WHERE product_id IN ({','.join(map(str, PROD_TARGET[:60]))})"
    densa = con.execute(f"""
    WITH cal AS (SELECT DISTINCT periodo FROM crudo),
         vida AS (SELECT product_id, min(periodo) p0, max(periodo) p1 FROM crudo
                  {filtro} GROUP BY 1),
         ini  AS (SELECT customer_id, min(periodo) c0 FROM crudo GROUP BY 1)
    SELECT i.customer_id, v.product_id, c.periodo, coalesce(s.tn, 0.0) AS tn
    FROM vida v
    CROSS JOIN ini i
    JOIN cal c ON c.periodo BETWEEN v.p0 AND v.p1 AND c.periodo >= i.c0
    LEFT JOIN crudo s ON s.customer_id = i.customer_id
                     AND s.product_id = v.product_id
                     AND s.periodo = c.periodo
    """).pl()
    prods = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
             .select(["product_id", "cat2", "cat3", "brand", "sku_size"])
             .unique(subset=["product_id"]))
    densa = (densa.join(prods, on="product_id", how="left")
             .with_columns((pl.col("customer_id").cast(pl.Utf8) + "_"
                            + pl.col("product_id").cast(pl.Utf8)).alias("par"))
             .sort(["par", "periodo"]))
    escribir_atomico(lambda p: densa.write_parquet(p), CK_DENSA)
    print(f"densa: {densa.height:,} filas en {time.time() - t0:.0f}s -> {CK_DENSA}")

_m = densa.group_by("par").agg([pl.len().alias("n"), pl.col("periodo").min().alias("a"),
                                pl.col("periodo").max().alias("b")])
_m = _m.with_columns(((pl.col("b") // 100 - pl.col("a") // 100) * 12
                      + (pl.col("b") % 100 - pl.col("a") % 100) + 1).alias("esp"))
assert (_m["n"] == _m["esp"]).all(), "hay huecos de periodo: shift(k) dejaria de ser el lag k"
print(f"grid contiguo verificado | pares: {densa['par'].n_unique():,} | "
      f"productos: {densa['product_id'].n_unique():,}")
if not PARAM["modo_test"]:
    _esp = 17_173_448   # criterio catedra z601, cartesiano completo
    if densa.height != _esp:
        print(f"\n>>> ATENCION: la densa tiene {densa.height:,} filas y el criterio catedra da "
              f"{_esp:,}.\n>>> Si dice 10,469,949 estas levantando un checkpoint viejo (solo pares "
              f"observados).\n>>> Subi PARAM['version_datos'] o borra {DIR_CK}.\n")
del _m; gc.collect()

## 3 — Escalado y feature engineering

`s(t)` es **relativo al mes donde estás parado**: media expandida de `tn` hasta t **inclusive**.
Nunca toca `t+1` ni `t+2`. Las alternativas (`media_movil_12`, `mediana_movil_12`) son también
ventanas que terminan en t.

Regla de oro de polars: el `.over(par)` va **al final** de la cadena. Si va adentro del `shift`,
la ventana móvil corre sobre la columna entera y se derrama de la cola de la serie anterior — eso
es fuga, y era un bug real del z702.

In [ ]:
def _escala(d: pl.DataFrame, g: str = "par") -> pl.DataFrame:
    tn = pl.col("tn")
    if PARAM["escala"] == "media_movil_12":
        e = tn.rolling_mean(12, min_samples=1).over(g)
    elif PARAM["escala"] == "mediana_movil_12":
        e = tn.rolling_median(12, min_samples=1).over(g)
    else:
        e = tn.cum_sum().over(g) / pl.int_range(1, pl.len() + 1).over(g)
    return d.with_columns(e.alias("escala"))


def _fe(d: pl.DataFrame) -> pl.DataFrame:
    g = "par"
    tn = pl.col("tn")
    d = _escala(d, g)
    esc = pl.max_horizontal(pl.col("escala"), pl.lit(PARAM["piso_escala"]))

    d = d.with_columns([
        (tn / esc).alias("nivel_rel"),
        (tn.shift(1).over(g) / esc).alias("lag1_rel"),
        (tn.shift(2).over(g) / esc).alias("lag2_rel"),
        (tn.shift(3).over(g) / esc).alias("lag3_rel"),
        (tn.shift(12).over(g) / esc).alias("lag12_rel"),
        (tn.shift(1).rolling_mean(3, min_samples=1).over(g) / esc).alias("rmean3_rel"),
        (tn.shift(1).rolling_mean(12, min_samples=1).over(g) / esc).alias("rmean12_rel"),
        (tn.shift(1).rolling_std(3, min_samples=2).over(g) / esc).alias("rstd3_rel"),
        ((tn > 0).cast(pl.Int32).rolling_sum(12, min_samples=1).over(g)).alias("ventas_ult12"),
        (pl.col("periodo") % 100).alias("mes"),
        pl.col("escala").log1p().alias("log_escala"),
        (tn > 0).cast(pl.Int32).alias("_hay"),
    ])
    d = d.with_columns(
        (pl.col("rmean3_rel") / (pl.col("rmean12_rel") + 1e-6)).alias("tendencia_3_12"))

    # recencia de compra: posicion dentro del bloque que arranca en cada compra
    d = d.with_columns(pl.col("_hay").cum_sum().over(g).alias("_bloque"))
    d = d.with_columns(pl.int_range(1, pl.len() + 1).over([g, "_bloque"]).alias("_pos"))
    d = d.with_columns(
        pl.when(pl.col("_bloque") == 0).then(pl.col("_pos"))
          .otherwise(pl.col("_pos") - 1).alias("meses_desde_compra"))

    # contexto de producto, cliente y familia en el mismo periodo
    d = d.with_columns([
        pl.col("tn").sum().over(["product_id", "periodo"]).alias("_tn_prod"),
        pl.col("tn").sum().over(["customer_id", "periodo"]).alias("_tn_cli"),
        pl.col("tn").sum().over(["cat3", "periodo"]).alias("_tn_cat3"),
        pl.col("tn").sum().over(["cat3", "brand", "customer_id", "periodo"]).alias("_tn_fam"),
    ])
    d = d.with_columns([
        (pl.col("_tn_prod").shift(1).over(g) / esc).alias("tn_prod_rel"),
        (tn / (pl.col("_tn_prod") + 1e-9)).alias("share_par_en_prod"),
        (tn / (pl.col("_tn_cli") + 1e-9)).alias("share_par_en_cli"),
        (pl.col("_tn_prod") / (pl.col("_tn_cat3") + 1e-9)).alias("share_prod_en_cat3"),
        (pl.col("_tn_fam").rolling_sum(3, min_samples=1).over(g) / (3 * esc))
            .alias("stock_idx_familia_3"),
        (pl.col("_tn_prod").shift(1).rolling_mean(3, min_samples=1).over(g)
         / (pl.col("_tn_prod").shift(1).rolling_mean(12, min_samples=1).over(g) + 1e-6)
         ).alias("tendencia_prod_3_12"),
    ])

    # rank del cliente por volumen acumulado hasta t. Se calcula sobre una tabla cliente x periodo
    # ORDENADA POR PERIODO: un cum_sum().over(customer_id) sobre el frame par-nivel sigue el orden
    # de filas, no el temporal, y el test de causalidad lo caza.
    cli = (d.group_by(["customer_id", "periodo"]).agg(pl.col("tn").sum().alias("_v"))
           .sort(["customer_id", "periodo"])
           .with_columns(pl.col("_v").cum_sum().over("customer_id").alias("_acum")))
    cli = cli.with_columns((pl.col("_acum").rank("average").over("periodo")
                            / pl.len().over("periodo")).alias("cliente_rank_rel"))
    d = d.join(cli.select(["customer_id", "periodo", "cliente_rank_rel"]),
               on=["customer_id", "periodo"], how="left")

    d = _estacionalidad(d)
    return d.drop([c for c in d.columns if c.startswith("_")]).sort(["par", "periodo"])


def _estacionalidad(d: pl.DataFrame) -> pl.DataFrame:
    """Estacionalidad del MES OBJETIVO y avance anual del producto. Todo con datos `<= t`.

    Por que importa: el sell-in tiene indice 1.15 en octubre y 0.84 en diciembre. Con horizonte 2,
    el modelo parado en octubre ve su propio nivel inflado y tiene que predecir el piso del ano.
    Sin decirle cuanto pesa historicamente el mes que va a predecir, sobre-predice ~37%.

    La causalidad la garantiza el `join_asof` hacia atras: para una fila de `t` que apunta al mes
    `m*`, toma la ultima ocurrencia de `m*` con `periodo <= t`. Si `m*` todavia no ocurrio nunca,
    queda 1.0 (neutro).
    """
    h = PARAM["horizonte"]
    d = d.with_columns(((((pl.col("periodo") % 100) - 1 + h) % 12) + 1).cast(pl.Int32)
                       .alias("mes_objetivo"))

    pp = (d.group_by(["product_id", "periodo"]).agg(pl.col("tn").sum().alias("tnp"))
          .with_columns([(pl.col("periodo") % 100).cast(pl.Int32).alias("mes"),
                         (pl.col("periodo") // 100).cast(pl.Int32).alias("anio")]))

    # media expandida del producto para ESE mes calendario, y media expandida global
    pm = (pp.sort(["product_id", "mes", "periodo"])
          .with_columns((pl.col("tnp").cum_sum().over(["product_id", "mes"])
                         / pl.int_range(1, pl.len() + 1).over(["product_id", "mes"]))
                        .alias("_m_mes"))
          .select(["product_id", "mes", "periodo", "_m_mes"]).sort("periodo"))
    pg = (pp.sort(["product_id", "periodo"])
          .with_columns((pl.col("tnp").cum_sum().over("product_id")
                         / pl.int_range(1, pl.len() + 1).over("product_id")).alias("_m_glob"))
          .select(["product_id", "periodo", "_m_glob"]))

    izq = (d.select(["product_id", "periodo", "mes_objetivo"]).unique()
           .with_columns(pl.col("mes_objetivo").alias("mes")).sort("periodo"))
    est = (izq.join_asof(pm, on="periodo", by=["product_id", "mes"], strategy="backward")
           .join(pg, on=["product_id", "periodo"], how="left")
           .with_columns((pl.col("_m_mes") / (pl.col("_m_glob") + 1e-9))
                         .fill_null(1.0).fill_nan(1.0).alias("estacional_prod_objetivo"))
           .select(["product_id", "periodo", "estacional_prod_objetivo"]))

    # avance anual del producto: acumulado del ano hasta t contra el mismo tramo del ano anterior
    ytd = (pp.sort(["product_id", "anio", "mes"])
           .with_columns(pl.col("tnp").cum_sum().over(["product_id", "anio"]).alias("_ytd")))
    prev = ytd.select([pl.col("product_id"), (pl.col("anio") + 1).alias("anio"),
                       pl.col("mes"), pl.col("_ytd").alias("_ytd_prev")])
    ytd = (ytd.join(prev, on=["product_id", "anio", "mes"], how="left")
           .with_columns((pl.col("_ytd") / (pl.col("_ytd_prev") + 1e-9))
                         .fill_null(1.0).fill_nan(1.0).alias("avance_anual_prod"))
           .select(["product_id", "periodo", "avance_anual_prod"]))

    return (d.join(est, on=["product_id", "periodo"], how="left")
            .join(ytd, on=["product_id", "periodo"], how="left")
            .with_columns([pl.col("estacional_prod_objetivo").fill_null(1.0),
                           pl.col("avance_anual_prod").fill_null(1.0)]))


CK_FE = DIR_CK / "fe.parquet"
if CK_FE.exists():
    datos = pl.read_parquet(CK_FE)
    faltan = [c for c in FEATS if c not in datos.columns]
    if faltan:
        raise RuntimeError(
            f"El checkpoint de FE no tiene {faltan}. Subi PARAM['version_datos'] "
            f"o borra {CK_FE}.")
    print(f"FE desde checkpoint {CK_FE.parent.name}: {datos.height:,} filas")
else:
    t0 = time.time()
    datos = _fe(densa).with_columns(
        pl.col("tn").shift(-PARAM["horizonte"]).over("par").alias("clase"))
    cols = ["par", "customer_id", "product_id", "periodo", "tn", "escala", "clase"] + FEATS
    datos = datos.select(list(dict.fromkeys(cols)))
    escribir_atomico(lambda p: datos.write_parquet(p), CK_FE)
    print(f"FE: {datos.height:,} filas x {len(datos.columns)} cols en {time.time() - t0:.0f}s")

## 4 — Test de causalidad (el notebook aborta si falla)

Recalcula el FE **entero** sobre la historia truncada en T y compara contra el FE calculado sobre
toda la historia y después filtrado a T. Si una sola feature difiere, esa feature usó información
de después de T. No hay excusa posible: aborta.

In [ ]:
def test_causalidad(T: int = 201806, n_pares: int = 800):
    pares = densa["par"].unique().sort().head(n_pares).to_list()
    sub = densa.filter(pl.col("par").is_in(pares))
    a = _fe(sub).filter(pl.col("periodo") <= T).sort(["par", "periodo"])
    b = _fe(sub.filter(pl.col("periodo") <= T)).sort(["par", "periodo"])
    assert a.height == b.height, f"distinta cantidad de filas: {a.height} vs {b.height}"
    malas = []
    for c in FEATS + ["escala"]:
        x, y = a[c], b[c]
        if x.dtype == pl.Utf8 or c in CATEGORICAS:
            if not (x.to_numpy() == y.to_numpy()).all():
                malas.append(c)
            continue
        xn, yn = x.to_numpy().astype(float), y.to_numpy().astype(float)
        if not np.allclose(np.nan_to_num(xn), np.nan_to_num(yn), rtol=1e-9, atol=1e-9):
            malas.append(c)
    if malas:
        raise AssertionError(
            f"FUGA DE FUTURO en: {malas}. El FE de esas columnas usa informacion de despues de T.")
    print(f"test de causalidad OK: {len(FEATS) + 1} columnas, T={T}, {n_pares} pares")


test_causalidad()
test_causalidad(T=201903, n_pares=800)

## 5 — Clusters DTW a nivel producto

Se clusteriza la serie mensual de cada **producto** (agregando clientes), que es el nivel al que
mide Kaggle. Los pares heredan el cluster de su producto: el dataset queda particionado, y como
ningún producto se reparte entre clusters, `Σ_p |e_p|` se descompone exacto por cluster.

Ventana común de 24 meses con padding a izquierda para que la banda Sakoe-Chiba de 3 sea honesta
(el z702 usaba `max(3, |Δlargo|)`, que en pares de largos dispares se abría hasta desactivarse).
`k` se elige por silhouette con restricción de balance sobre `[4..8]` — la cátedra espera 6/7,
pero se elige, no se hardcodea.

**Se recalcula en cada corte temporal.** Si se calculara una vez sobre toda la serie, la etiqueta
de cluster de una fila de 2018 estaría condicionada por lo que pasó en 2019: fuga sutil y real.

In [ ]:
def _series_producto(corte: int) -> tuple[list, np.ndarray]:
    d = (datos.filter(pl.col("periodo") <= corte)
         .group_by(["product_id", "periodo"]).agg(pl.col("tn").sum())
         .sort(["product_id", "periodo"]))
    prods, M = [], []
    W = PARAM["cl_ventana"]
    for pid, g in d.group_by("product_id", maintain_order=True):
        v = g["tn"].to_numpy()[-W:]
        if len(v) < W:
            v = np.concatenate([np.zeros(W - len(v)), v])
        s = v.std()
        M.append((v - v.mean()) / (s if s > 1e-9 else 1.0))   # z-score: DTW compara FORMA
        prods.append(int(pid[0]) if isinstance(pid, tuple) else int(pid))
    return prods, np.asarray(M, dtype=np.double)


def _matriz_dtw(M: np.ndarray) -> np.ndarray:
    n = len(M)
    if dtw is not None:
        D = dtw.distance_matrix_fast(M, window=PARAM["cl_banda"],
                                     parallel=True, compact=False)
        iu = np.triu_indices(n, 1)
        v = np.asarray(D)[iu]
        finito = np.isfinite(v)
        if not finito.all():                    # banda infactible -> sanear, no propagar inf
            v = np.where(finito, v, (v[finito].max() if finito.any() else 1.0) * 10)
        F = np.zeros((n, n)); F[iu] = v
        return F + F.T
    print("   dtaidistance no disponible -> distancia euclidea sobre la forma (fallback)")
    return np.sqrt(((M[:, None, :] - M[None, :, :]) ** 2).sum(-1))


def clusters_en(corte: int) -> dict:
    ck = DIR_CK / f"clusters_{corte}.json"
    if ck.exists():
        return {int(k): v for k, v in json.loads(ck.read_text()).items()}
    if not PARAM["usar_clusters"]:
        prods, _ = _series_producto(corte)
        return dict.fromkeys(prods, 0)
    t0 = time.time()
    prods, M = _series_producto(corte)
    D = _matriz_dtw(M)
    Z = linkage(squareform(D, checks=False), method="average")
    cands = []
    for k in PARAM["cl_lista_k"]:
        lab = fcluster(Z, k, criterion="maxclust")
        if len(set(lab)) < 2:
            continue
        bal = np.bincount(lab)[1:].min() / len(lab)
        cands.append({"k": k, "lab": lab, "bal": bal,
                      "sil": silhouette_score(D, lab, metric="precomputed")})
    ok = [c for c in cands if c["bal"] >= PARAM["cl_balance_min"]]
    best = max(ok, key=lambda c: c["sil"]) if ok else max(cands, key=lambda c: c["bal"])
    mapa = {p: int(l) - 1 for p, l in zip(prods, best["lab"])}
    escribir_atomico(lambda f: Path(f).write_text(json.dumps({str(k): v for k, v in mapa.items()})), ck)
    print(f"   corte {corte}: k={best['k']} silhouette={best['sil']:.3f} "
          f"balance={best['bal']:.3f} tamanos={np.bincount(best['lab'])[1:].tolist()} "
          f"({time.time() - t0:.0f}s)")
    return mapa


CORTES = sorted({periodo_menos(a, PARAM["horizonte"])
                 for a in PARAM["anclas_optuna"] + PARAM["anclas_reporte"]}
                | {PARAM["periodo_inferencia"]})
print("calculando clusters por corte (causal):")
CLUSTERS = {c: clusters_en(c) for c in CORTES}
K = max(max(m.values()) for m in CLUSTERS.values()) + 1
print(f"K = {K}")

## 6 — Matrices, entrenamiento y métrica

**El peso.** `w = s^(p−1)`, y para las filas de pares dormidos submuestreadas al 25 %, un factor
`1/0,25` que las devuelve a su peso original — el estimador sigue siendo el de la misma pérdida,
con la memoria dividida.

**La métrica** es WAPE agregando a **producto**: se suman las predicciones de todos los pares del
producto *antes* de medir. Los errores entre clientes del mismo producto se cancelan, igual que
en Kaggle.

In [ ]:
COD = {c: {v: i for i, v in enumerate(sorted(datos[c].unique().drop_nulls().to_list()))}
       for c in CATEGORICAS}
print("categoricas:", {c: len(v) for c, v in COD.items()})
CAT_NOM = list(CATEGORICAS)


def paquete(corte: int, modo: str, semilla: int, mapa: dict) -> dict:
    """Parado en `corte` (ultimo mes con datos), con horizonte h.

    modo='train': filas con `periodo <= corte - h`. Es la unica ventana cuya clase `tn(t+h)`
        ya ocurrio en `<= corte`. Si se tomara `periodo <= corte`, las filas de los ultimos h
        meses traerian como clase justamente el mes que se quiere predecir: fuga directa.
    modo='eval' : filas de `periodo == corte`, cuya clase es `tn(corte + h)` — el mes objetivo.
    """
    h = PARAM["horizonte"]
    d = (datos.filter(pl.col("periodo") == corte) if modo == "eval"
         else datos.filter((pl.col("periodo") <= periodo_menos(corte, h))
                           & pl.col("clase").is_not_null()))
    cols = list(dict.fromkeys(["product_id", "periodo", "escala", "clase", "ventas_ult12"] + FEATS))
    pdf = d.select(cols).to_pandas()
    for c in CATEGORICAS:
        if c in pdf.columns:
            pdf[c] = pdf[c].map(COD[c]).fillna(-1).astype("int32")

    s = np.maximum(pdf["escala"].to_numpy(), PARAM["piso_escala"])
    y = pdf["clase"].to_numpy()
    prod = pdf["product_id"].to_numpy()
    p = PARAM["tweedie_power"]
    w = s ** (p - 1.0) if PARAM["usar_peso_escala"] else np.ones_like(s)

    if modo == "train":
        # el corte ya garantiza que ninguna clase de entrenamiento cae despues de `corte`
        assert pdf["periodo"].max() + 0 <= periodo_menos(corte, PARAM["horizonte"]), \
            "hay filas de train cuya clase cae despues del corte"
        if PARAM["submuestreo_dormidos"] < 1.0:
            rng = np.random.default_rng(semilla)
            dormida = (pdf["ventas_ult12"].to_numpy() == 0) & (y == 0)
            tasa = PARAM["submuestreo_dormidos"]
            keep = ~dormida | (rng.random(len(y)) < tasa)
            w = np.where(dormida, w / tasa, w)
            pdf, y, s, w, prod = pdf[keep], y[keep], s[keep], w[keep], prod[keep]
        if PARAM["peso_febrero"] > 1:
            w = w * np.where(pdf["periodo"].to_numpy() % 100 == 2, PARAM["peso_febrero"], 1.0)

    cl = np.array([mapa.get(int(x), 0) for x in prod], dtype=np.int16)
    return {"X": pdf[FEATS].to_numpy(dtype=np.float32), "y": y, "s": s, "w": w,
            "prod": prod, "per": pdf["periodo"].to_numpy(), "cl": cl}


def dataset(pack: dict, idx: np.ndarray) -> lgb.Dataset:
    ds = lgb.Dataset(pack["X"][idx], label=pack["y"][idx], weight=pack["w"][idx],
                     init_score=np.log(pack["s"][idx]),
                     feature_name=FEATS, categorical_feature=CAT_NOM,
                     free_raw_data=True,
                     params={"max_bin": PARAM["max_bin"], "verbosity": -1})
    ds.construct()
    return ds


def predecir(bst, X, s) -> np.ndarray:
    f = np.clip(bst.predict(X, raw_score=True), -30, 30)
    return np.maximum(s * np.exp(f), 0.0)


def wape_producto(y, pred, prod) -> float:
    df = pd.DataFrame({"p": prod, "y": y, "h": pred}).groupby("p", sort=False).sum()
    den = df["y"].sum()
    return float(np.abs(df["y"] - df["h"]).sum() / den) if den > 0 else float("nan")


def grupos(prod: np.ndarray, per: np.ndarray, y: np.ndarray):
    """Indice compacto de ⟨producto, periodo⟩ y el tonelaje real agregado de cada grupo."""
    clave = prod.astype(np.int64) * 1000000 + per.astype(np.int64)
    _, inv = np.unique(clave, return_inverse=True)
    return inv.astype(np.int32), np.bincount(inv, weights=y)


def objetivo_agrupado(inv: np.ndarray, y_grupo: np.ndarray):
    """Gradiente = signo del error AGREGADO por ⟨producto, mes⟩. Es el numerador de WAPE.

    La pérdida por fila penaliza errores que la métrica perdona: si un par sobra 10 t y otro del
    mismo producto falta 10 t, Kaggle no cobra nada. Acá los errores se cancelan ANTES de
    penalizar, así que el modelo deja de gastar capacidad en ruido que no se mide.

    Requiere el agregado completo: con este objetivo se apagan el submuestreo de dormidos y el
    peso `s^(p−1)` (que existe para alinear la pérdida por fila, y acá ya no hace falta).
    """
    def fobj(preds, ds):
        mu = np.exp(np.clip(preds, -30, 30))
        agg = np.bincount(inv, weights=mu, minlength=len(y_grupo))
        sg = np.sign(agg - y_grupo)[inv]
        return mu * sg, mu + 1e-6
    return fobj


def fobj_de(g):
    """Devuelve el objetivo custom si el modo es 'agrupado'; si no, None (Tweedie nativo)."""
    if PARAM["objetivo"] != "agrupado" or g is None:
        return None
    return objetivo_agrupado(*g)


def entrenar(dtr, params, n, fobj=None):
    """Sin early stopping a proposito: el numero de arboles es un hiperparametro mas de Optuna.

    Parar por el fold de validacion lo usaria dos veces (para elegir n y para reportar el WAPE)
    y el numero que sale de ahi es optimista.
    """
    p = dict(params)
    if fobj is not None:
        p["objective"] = fobj
    return lgb.train(p, dtr, num_boost_round=n)


def params_base(trial=None, semilla=102191) -> dict:
    p = {"objective": "tweedie", "tweedie_variance_power": PARAM["tweedie_power"],
         "learning_rate": 0.03, "num_leaves": 127, "min_data_in_leaf": 300,
         "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1,
         "lambda_l1": 0.0, "lambda_l2": 0.0,
         "max_bin": PARAM["max_bin"], "num_threads": N_CORES, "verbosity": -1,
         "seed": semilla, "force_row_wise": True}
    if trial is not None:
        p.update({
            "tweedie_variance_power": trial.suggest_float("tvp", 1.1, 1.6),
            "learning_rate": trial.suggest_float("lr", 0.015, 0.08, log=True),
            "num_leaves": trial.suggest_int("leaves", 31, 511, log=True),
            "min_data_in_leaf": trial.suggest_int("mdl", 100, 3000, log=True),
            "feature_fraction": trial.suggest_float("ff", 0.5, 1.0),
            "bagging_fraction": trial.suggest_float("bf", 0.6, 1.0),
            "lambda_l1": trial.suggest_float("l1", 1e-4, 10.0, log=True),
            "lambda_l2": trial.suggest_float("l2", 1e-4, 10.0, log=True),
        })
    return p

## 7 — Folds

Un fold entrena hasta `ancla − 2` y evalúa en el ancla: el horizonte real del deploy. Los
`lgb.Dataset` se construyen **una vez por (fold, cluster)** y se reusan en todos los trials de
Optuna — con `max_bin=1023` el binning es lo caro, y la interfaz sklearn lo rehace en cada `fit`.

In [ ]:
FOLDS = []
for ancla in PARAM["anclas_optuna"]:
    corte = periodo_menos(ancla, PARAM["horizonte"])
    FOLDS.append({"ancla": ancla, "corte": corte, "mapa": CLUSTERS[corte],
                  "es_febrero": ancla % 100 == 2})
print("folds:", [(f["ancla"], f["corte"]) for f in FOLDS])

t0 = time.time()
for f in FOLDS:
    sem = PARAM["semillas_ensemble"][0]
    tr = paquete(f["corte"], "train", sem, f["mapa"])
    ev = paquete(f["corte"], "eval", sem, f["mapa"])
    f["ds"], f["grp"] = {}, {}
    for c in range(K):
        idx = np.where(tr["cl"] == c)[0]
        if len(idx) == 0:
            continue
        f["ds"][c] = dataset(tr, idx)
        f["grp"][c] = grupos(tr["prod"][idx], tr["per"][idx], tr["y"][idx])
    f["ev"] = ev
    f["n_tr"] = len(tr["y"])
    del tr; gc.collect()
    print(f"  ancla {f['ancla']}: train {f['n_tr']:,} filas, "
          f"eval {len(f['ev']['y']):,}, clusters {sorted(f['ds'])}")
print(f"datasets construidos en {time.time() - t0:.0f}s")

## 8 — Optuna, un estudio por cluster

Como cada producto vive en un solo cluster, `Σ_p |e_p|` se descompone exacto: optimizar el WAPE
**restringido a los productos del cluster** optimiza la métrica global. Eso es lo que habilita
entrenar y tunear por separado sin perder alineación con Kaggle.

Los folds de febrero pesan doble en el objetivo, porque el target es febrero.

In [ ]:
def evaluar_cluster(c: int, params: dict, n_arboles: int) -> float:
    num, den = 0.0, 0.0
    for f in FOLDS:
        if c not in f["ds"]:
            continue
        b = entrenar(f["ds"][c], params, n_arboles, fobj_de(f["grp"].get(c)))
        ev, m = f["ev"], f["ev"]["cl"] == c
        if m.sum() == 0:
            continue
        pred = predecir(b, ev["X"][m], ev["s"][m])
        df = pd.DataFrame({"p": ev["prod"][m], "y": ev["y"][m], "h": pred}).groupby("p").sum()
        peso = PARAM["peso_febrero"] if f["es_febrero"] else 1.0
        num += peso * np.abs(df["y"] - df["h"]).sum()
        den += peso * df["y"].sum()
        del b; gc.collect()
    return num / den if den > 0 else float("inf")


PISO_ARBOLES = max(40, min(150, PARAM["techo_arboles"] // 2))
CK_BEST = DIR_CK / "mejores_params.json"
if CK_BEST.exists():
    MEJORES = {int(k): v for k, v in json.loads(CK_BEST.read_text()).items()}
    print("hiperparametros desde checkpoint")
else:
    MEJORES = {}
    for c in range(K):
        if not any(c in f["ds"] for f in FOLDS):
            continue
        t0 = time.time()
        if HAY_OPTUNA:
            st = optuna.create_study(
                direction="minimize", study_name=f"c{c}",
                storage=f"sqlite:///{DIR_CK}/optuna.db", load_if_exists=True,
                sampler=optuna.samplers.TPESampler(seed=102191 + c))

            def obj(trial, c=c):
                n = trial.suggest_int("arboles", PISO_ARBOLES, PARAM["techo_arboles"], log=True)
                return evaluar_cluster(c, params_base(trial), n)

            st.optimize(obj, n_trials=PARAM["n_trials"], show_progress_bar=False)
            MEJORES[c] = {"params": {**params_base(), **{
                "tweedie_variance_power": st.best_params["tvp"],
                "learning_rate": st.best_params["lr"],
                "num_leaves": st.best_params["leaves"],
                "min_data_in_leaf": st.best_params["mdl"],
                "feature_fraction": st.best_params["ff"],
                "bagging_fraction": st.best_params["bf"],
                "lambda_l1": st.best_params["l1"], "lambda_l2": st.best_params["l2"]}},
                "n_arboles": st.best_params["arboles"],
                "wape_val": st.best_value}
        else:
            n = PARAM["techo_arboles"] // 2
            MEJORES[c] = {"params": params_base(), "n_arboles": n,
                          "wape_val": evaluar_cluster(c, params_base(), n)}
        print(f"  cluster {c}: WAPE val {MEJORES[c]['wape_val']:.4f}, "
              f"{MEJORES[c]['n_arboles']} arboles ({time.time() - t0:.0f}s)")
    escribir_atomico(lambda p: Path(p).write_text(
        json.dumps({str(k): v for k, v in MEJORES.items()}, indent=2, default=str)), CK_BEST)

wape_global = float(np.average([m["wape_val"] for m in MEJORES.values()]))
print(f"\nWAPE de validacion (promedio de clusters): {wape_global:.4f}")

## 9 — Reporte mes a mes y multiplicador

Acá se ve si el modelo **generaliza a cualquier mes** o si sólo sabe hacer febrero. Se entrena con
los hiperparámetros elegidos y se mide en las siete anclas, una por una.

El ancla a mirar es **201812**: su origen es octubre, el segundo pico del año, y el target es
diciembre, el piso. Es el fold donde un modelo sin corrección estacional se estrella. Si ahí el
WAPE se dispara o el ratio `y/ŷ` cae muy por debajo de 1, `estacional_prod_objetivo` no está
haciendo su trabajo.

El multiplicador `m* = argmin_m Σ|y − m·ŷ|` es la **mediana ponderada** de `y_p/ŷ_p` con pesos
`ŷ_p`. Se estima en estos folds —nunca en el leaderboard— y se encoge hacia 1 con λ=0,7. Se
reporta el de cada mes: si varían mucho entre sí, un multiplicador global no es la herramienta.

In [ ]:
def multiplicador(y, yhat) -> float:
    ok = yhat > 0
    if ok.sum() == 0:
        return 1.0
    r, w = y[ok] / yhat[ok], yhat[ok]
    o = np.argsort(r); r, w = r[o], w[o]
    return float(r[np.searchsorted(np.cumsum(w), 0.5 * w.sum())])


def predecir_fold(corte: int, mapa: dict) -> tuple:
    tr = paquete(corte, "train", PARAM["semillas_ensemble"][0], mapa)
    ev = paquete(corte, "eval", PARAM["semillas_ensemble"][0], mapa)
    pred = np.zeros(len(ev["y"]))
    for c, m in MEJORES.items():
        itr, iev = np.where(tr["cl"] == c)[0], np.where(ev["cl"] == c)[0]
        if len(itr) == 0 or len(iev) == 0:
            continue
        d = dataset(tr, itr)
        b = entrenar(d, m["params"], m["n_arboles"],
                     fobj_de(grupos(tr["prod"][itr], tr["per"][itr], tr["y"][itr])))
        pred[iev] = predecir(b, ev["X"][iev], ev["s"][iev])
        del d, b; gc.collect()
    del tr; gc.collect()
    return ev, pred


REPORTE = []
for ancla in PARAM["anclas_reporte"]:
    corte = periodo_menos(ancla, PARAM["horizonte"])
    t0 = time.time()
    ev, pred = predecir_fold(corte, CLUSTERS[corte])
    df = pd.DataFrame({"p": ev["prod"], "y": ev["y"], "h": pred}).groupby("p").sum()
    REPORTE.append({"ancla": ancla, "origen": corte,
                    "wape": float(np.abs(df["y"] - df["h"]).sum() / df["y"].sum()),
                    "m": multiplicador(df["y"].to_numpy(), df["h"].to_numpy()),
                    "real_tn": float(df["y"].sum()), "pred_tn": float(df["h"].sum())})
    print(f"  ancla {ancla} (origen {corte}): WAPE {REPORTE[-1]['wape']:.4f}  "
          f"m*={REPORTE[-1]['m']:.3f}  real {df['y'].sum():,.0f} vs pred {df['h'].sum():,.0f} tn  "
          f"({time.time() - t0:.0f}s)")
    del ev; gc.collect()

rep = pd.DataFrame(REPORTE)
rep["mes"] = rep["ancla"] % 100
print("\n" + rep[["ancla", "origen", "mes", "wape", "m", "real_tn", "pred_tn"]].round(4).to_string(index=False))
print(f"\nWAPE medio {rep['wape'].mean():.4f}  |  peor mes: ancla "
      f"{int(rep.loc[rep['wape'].idxmax(), 'ancla'])} con {rep['wape'].max():.4f}")
_oct = rep[rep.ancla == 201812]
if len(_oct):
    print(f"fold octubre->diciembre: WAPE {_oct['wape'].iloc[0]:.4f}, m*={_oct['m'].iloc[0]:.3f} "
          f"(m* muy por debajo de 1 = sigue sobre-prediciendo el pico de octubre)")

# --- A/B: ¿la estacionalidad del mes objetivo paga? ---
# Mismos hiperparametros, mismos folds, mismas semillas: lo unico que cambia son las 3 features.
# Se compara PAREADO por mes, que es mucho mas sensible que comparar promedios (cada mes tiene su
# propia dificultad y al restar se cancela).
SEASONALES = [f for f in ("mes_objetivo", "estacional_prod_objetivo", "avance_anual_prod")
              if f in FEATS]
if PARAM["ab_estacional"] and SEASONALES:
    _base_f, _base_c = list(FEATS), list(CAT_NOM)
    FEATS = [f for f in _base_f if f not in SEASONALES]
    CAT_NOM = [c for c in _base_c if c in FEATS]
    print(f"\nA/B sin estacionalidad ({len(FEATS)} features, se sacan {SEASONALES}):")
    sin = []
    for ancla in PARAM["anclas_reporte"]:
        corte = periodo_menos(ancla, PARAM["horizonte"])
        ev, pred = predecir_fold(corte, CLUSTERS[corte])
        df = pd.DataFrame({"p": ev["prod"], "y": ev["y"], "h": pred}).groupby("p").sum()
        sin.append({"ancla": ancla,
                    "wape_sin": float(np.abs(df["y"] - df["h"]).sum() / df["y"].sum()),
                    "m_sin": multiplicador(df["y"].to_numpy(), df["h"].to_numpy())})
        print(f"  ancla {ancla}: WAPE {sin[-1]['wape_sin']:.4f}  m*={sin[-1]['m_sin']:.3f}")
        del ev; gc.collect()
    FEATS, CAT_NOM = _base_f, _base_c

    ab = rep.merge(pd.DataFrame(sin), on="ancla")
    ab["delta"] = ab["wape"] - ab["wape_sin"]
    print("\n" + ab[["ancla", "wape_sin", "wape", "delta", "m_sin", "m"]].round(4).to_string(index=False))
    d, ee = ab["delta"].mean(), ab["delta"].std(ddof=1) / np.sqrt(len(ab))
    print(f"\ndelta medio {d:+.4f} +- {ee:.4f} (error est.)  |  mejora en {(ab.delta < 0).sum()}/{len(ab)} meses")
    print("veredicto:", "LA ESTACIONALIDAD PAGA" if d < -2 * ee else
          ("EMPEORA" if d > 2 * ee else "DENTRO DEL RUIDO: no hay evidencia de que sirva"))
else:
    ab = None

pesos = np.where(rep["mes"] == 2, PARAM["peso_febrero"], 1.0)
m_bar = float(np.average(rep["m"], weights=pesos))
M_FINAL = 1 + PARAM["lambda_shrink"] * (m_bar - 1)
print(f"\nm̄ (ponderando febrero x{PARAM['peso_febrero']}) = {m_bar:.3f}  ->  "
      f"m_final (shrink {PARAM['lambda_shrink']}) = {M_FINAL:.3f}")
print(f"dispersion de m* entre meses: {rep['m'].std():.3f} "
      f"({'homogeneo, un multiplicador sirve' if rep['m'].std() < 0.08 else 'ALTO: el sesgo cambia con el mes, desconfia del multiplicador global'})")

## 10 — Entrenamiento final e inferencia

Se entrena con **toda** la historia hasta 201912 y se predice 202002. Ensemble de 3 semillas por
cluster, promediado. Los clusters de inferencia son los calculados con corte 201912.

In [ ]:
mapa_inf = CLUSTERS[PARAM["periodo_inferencia"]]
tr = paquete(PARAM["periodo_inferencia"], "train", PARAM["semillas_ensemble"][0], mapa_inf)
inf = paquete(PARAM["periodo_inferencia"], "eval", PARAM["semillas_ensemble"][0], mapa_inf)
print(f"train final: {len(tr['y']):,} filas | inferencia: {len(inf['y']):,} filas")

pred = np.zeros(len(inf["y"]))
for c, m in MEJORES.items():
    itr, iev = np.where(tr["cl"] == c)[0], np.where(inf["cl"] == c)[0]
    if len(itr) == 0 or len(iev) == 0:
        continue
    t0 = time.time()
    d = dataset(tr, itr)
    fobj_fin = fobj_de(grupos(tr["prod"][itr], tr["per"][itr], tr["y"][itr]))
    acum = np.zeros(len(iev))
    for sem in PARAM["semillas_ensemble"]:
        b = entrenar(d, {**m["params"], "seed": sem}, m["n_arboles"], fobj_fin)
        acum += predecir(b, inf["X"][iev], inf["s"][iev])
        del b; gc.collect()
    pred[iev] = acum / len(PARAM["semillas_ensemble"])
    del d; gc.collect()
    print(f"  cluster {c}: {len(itr):,} filas train, {len(iev):,} de inferencia "
          f"({time.time() - t0:.0f}s)")

del tr; gc.collect()

sub = (pd.DataFrame({"product_id": inf["prod"], "tn": pred})
       .groupby("product_id", as_index=False).sum())
sub = sub[sub["product_id"].isin(PROD_TARGET)].sort_values("product_id").reset_index(drop=True)
faltan = set(PROD_TARGET) - set(sub["product_id"])
if faltan:
    sub = pd.concat([sub, pd.DataFrame({"product_id": sorted(faltan), "tn": 0.0})]
                    ).sort_values("product_id").reset_index(drop=True)
print(f"\nsubmit: {len(sub)} productos ({len(faltan)} sin prediccion, van en 0)")

## 11 — Test de nivel y submits

La banda **28 400 – 30 600 tn** sale de los febreros reales de los 780 (27 304 / 28 185 / 27 200
en 2017-19) corrigiendo por los 120 productos que no existían en feb-2019 y que hoy aportan
1 574 tn/mes. Es un chequeo de cordura, no un juez: si el total cae afuera, hay que mirar por qué
antes de subir. El z705 dio 26 588 y se subió igual.

In [ ]:
lo, hi = PARAM["banda_nivel"]
for nom, v in [("m=1.000", sub["tn"].values), (f"m={M_FINAL:.3f}", sub["tn"].values * M_FINAL)]:
    t = v.sum()
    estado = "DENTRO" if lo <= t <= hi else ("ALTO" if t > hi else "BAJO")
    print(f"  {nom}: total {t:,.0f} tn -> {estado} de la banda [{lo:,}, {hi:,}]")

WAPE_REPORTE = float(rep["wape"].mean())
SUBMITS = []
for nom, mult in [("mfinal", M_FINAL), ("m1", 1.0)]:
    d = sub.copy(); d["tn"] = np.maximum(d["tn"] * mult, 0.0)
    f = DIR_OUT / f"z709_{nom}_m{mult:.3f}.csv"
    escribir_atomico(lambda p, d=d: d.to_csv(p, index=False), f)
    SUBMITS.append((f, f"z709 par-nivel offset+peso s^(p-1)+estacional, {len(FEATS)} feats, "
                       f"K={K}, WAPE val {wape_global:.4f} / reporte {WAPE_REPORTE:.4f}, m={mult:.3f}"))
    print(f"  {f.name}: {d['tn'].sum():,.0f} tn")

(DIR_EXP / "resumen.json").write_text(json.dumps({
    "wape_val_por_cluster": {str(k): v["wape_val"] for k, v in MEJORES.items()},
    "wape_val": wape_global, "reporte_por_mes": REPORTE,
    "ab_estacional": (ab.to_dict("records") if ab is not None else None),
    "m_bar": m_bar, "m_final": M_FINAL, "K": K,
    "features": FEATS, "total_tn": float(sub["tn"].sum() * M_FINAL),
    "param": {k: v for k, v in PARAM.items() if k != "features"}}, indent=2, default=str))


def kaggle_submit(comp, archivo: Path, msg: str):
    flag = archivo.with_suffix(".done")
    if flag.exists():
        print("ya subido:", archivo.name); return
    r = subprocess.run(["kaggle", "competitions", "submit", "-c", comp,
                        "-f", str(archivo), "-m", msg], capture_output=True, text=True)
    print(archivo.name, "->", (r.stdout or r.stderr).strip()[:180])
    if r.returncode == 0:
        flag.write_text(msg)


if PARAM["submit"]:
    for f, msg in SUBMITS:
        kaggle_submit(PARAM["kaggle_competition"], f, msg)
else:
    print("submit=False -> CSVs en", DIR_OUT)